# 06 · Attach MapMyCells labels

*sc-rnaseq-wf, re-instantiated step by step for `team-jakobsson` (`jakobsson-pmdbs-sn-rnaseq`)*

| | |
| :- | :- |
| WDL task | [`add_mapped_cell_types`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/workflows/cohort_analysis/cohort_analysis.wdl) |
| Script | [`transcriptional_phenotype`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/docker/sc_tools/scripts/main/transcriptional_phenotype) → `sc-transcriptional-phenotype` |

This step turns the MapMyCells results into a **confident cell type label** per cell and joins it to `obs`. Only confident calls are kept: if the correlation coefficient or the bootstrapping probability is below 0.5, the cell is labelled `Unknown`. scANVI (notebook 09) uses the confident labels to train and predicts a label for every `Unknown` cell.

The taxonomy levels are read from the results header, and decide which level becomes the label:

| Taxonomy | `phenotype` | Confidence (`rho`, `prob`) taken at |
| :- | :- | :- |
| Siletti (supercluster / cluster / subcluster) | supercluster; cluster `Splat_395` (DA VGLUT2 neurons, inside the mixed `Splatter` supercluster) → `Dopaminergic` | supercluster level (cluster level for `Splat_395`) |
| SEA-AD (class / subclass / supertype) | `GABAergic` / `Glutamatergic` for neurons, subclass for non-neuronal cells | class level for neurons, subclass level otherwise |

`phenotype` is MapMyCells' call; `cell_type` is the same call, or `Unknown` if it isn't confident. The level names are kept as well (`supercluster_name`, `cluster_name`, …).

`sc-annotate` (notebook 09) re-derives these labels from a results CSV with the same function, so a different reference can be swapped in at that step.

**Inputs**
- `${cohort_id}.normalized.h5ad`
- `${cohort_id}.mmc_otf_mapping.${mmc_taxonomy}.results.csv`

**Outputs** (in `cfg.RUN_DIR`, named as in the WDL)
- `${cohort_id}.mmc.h5ad`
- `${cohort_id}.mmc_results.parquet`

**Previous:** [05_process](05_process.ipynb) · **Next:** [07_integrate_scvi](07_integrate_scvi.ipynb)

In [ ]:
import wf_config as cfg
import pandas as pd
import scanpy as sc
from IPython.display import Image, display

cfg.RUN_DIR.mkdir(parents=True, exist_ok=True)
print("run directory:", cfg.RUN_DIR)

## 1. Run `sc-transcriptional-phenotype`

In [ ]:
mmc_csv = cfg.out(f"mmc_otf_mapping.{cfg.MMC_TAXONOMY}.results.csv")
cfg.run(
    "sc-transcriptional-phenotype",
    "--adata-input", cfg.out("normalized.h5ad"),
    "--workflow-name", cfg.WORKFLOW_NAME,
    "--mmc-results", mmc_csv,
    "--adata-output", cfg.out("mmc.h5ad"),
    "--output-cell-types-file", cfg.out("mmc_results.parquet"),
)

## 2. Confident labels

How many of MapMyCells' calls for each phenotype pass the 0.5 thresholds.

In [ ]:
labels = pd.read_parquet(cfg.out("mmc_results.parquet"))
print(f"{len(labels):,} cells; {100 * (labels['cell_type'] == 'Unknown').mean():.1f}% Unknown")
pd.crosstab(labels["phenotype"], labels["cell_type"] == "Unknown").set_axis(["confident", "Unknown"], axis=1)

## 3. Labels per sample

Check that the expected cell types are present in each sample before relying on them for scANVI.

In [ ]:
samples = sc.read_h5ad(cfg.out("mmc.h5ad"), backed="r").obs["sample"]
pd.crosstab(samples, labels.loc[samples.index, "cell_type"], normalize="index").mul(100).round(1)

**Previous:** [05_process](05_process.ipynb) · **Next:** [07_integrate_scvi](07_integrate_scvi.ipynb)